# Starting11 - Transfermarkt Scraper

Scrapes World Cup match lineups from Transfermarkt to generate quiz data for the Starting11 game mode.

**Data extracted:**
- Starting 11 players for each team
- Club affiliations at match time (with badge URLs)
- Jersey numbers
- Formation positions

In [11]:
# Setup and imports
import requests
from bs4 import BeautifulSoup
import re
import json
import os
import time

REQUEST_DELAY = 3  # seconds between requests (respect rate limits)
BASE_URL = "https://www.transfermarkt.com"
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
}

def fetch_page(url, delay=True):
    """Fetch a page with rate limiting and proper headers."""
    if delay:
        print(f"  Waiting {REQUEST_DELAY}s before request...")
        time.sleep(REQUEST_DELAY)
    
    full_url = url if url.startswith('http') else BASE_URL + url
    print(f"  Fetching: {full_url}")
    
    try:
        response = requests.get(full_url, headers=HEADERS, timeout=15)
        response.raise_for_status()
        return BeautifulSoup(response.text, 'html.parser')
    except Exception as e:
        print(f"  Error: {e}")
        return None

print("✅ Setup complete")

✅ Setup complete


In [12]:
# Core scraper function
def scrape_match_lineup(match_url):
    """
    Scrape a Transfermarkt match lineup page.
    URL format: /team1_team2/aufstellung/spielbericht/{match_id}
    
    Returns list of player dicts with jersey, name, player_id, club info.
    """
    soup = fetch_page(match_url, delay=True)
    if not soup:
        return None
    
    players = []
    
    # Find all rows with inline-table (player rows)
    inline_tables = soup.find_all('table', class_='inline-table')
    
    for inline_table in inline_tables:
        # Navigate up to outer row: inline-table -> td -> tr
        outer_td = inline_table.find_parent('td')
        if not outer_td:
            continue
        outer_row = outer_td.find_parent('tr')
        if not outer_row:
            continue
            
        cells = outer_row.find_all('td', recursive=False)
        if len(cells) < 4:
            continue
        
        # Cell 0: Jersey number
        jersey_text = cells[0].get_text(strip=True)
        jersey = int(jersey_text) if jersey_text.isdigit() else None
        
        # Cell 1: Player info
        player_link = cells[1].find('a', href=re.compile(r'/profil/spieler/\d+'))
        if not player_link:
            continue
            
        href = player_link.get('href', '')
        player_match = re.search(r'/([^/]+)/profil/spieler/(\d+)', href)
        if not player_match:
            continue
            
        player_slug = player_match.group(1)
        player_id = player_match.group(2)
        
        # Get player name from nearby link with title
        name_link = cells[1].find('a', href=re.compile(r'/leistungsdatendetails/'))
        player_name = name_link.get('title', '') if name_link else player_slug.replace('-', ' ').title()
        
        # Cell 3: Club info
        club_link = cells[3].find('a', href=re.compile(r'/startseite/verein/\d+'))
        if not club_link:
            continue
            
        club_href = club_link.get('href', '')
        club_match = re.search(r'/([^/]+)/startseite/verein/(\d+)', club_href)
        if not club_match:
            continue
            
        club_slug = club_match.group(1)
        club_id = club_match.group(2)
        club_name = club_link.get('title', '') or club_slug.replace('-', ' ').title()
        
        players.append({
            'jersey': jersey,
            'name': player_name,
            'player_id': player_id,
            'player_slug': player_slug,
            'club_name': club_name,
            'club_id': club_id,
            'club_slug': club_slug
        })
    
    return players

print("✅ Scraper function defined")

✅ Scraper function defined


In [13]:
# Formation position templates
FORMATIONS = {
    "4-3-3": {
        "positions": [
            {"role": "GK", "x": 50, "y": 92},
            {"role": "LB", "x": 15, "y": 72},
            {"role": "CB", "x": 35, "y": 75},
            {"role": "CB", "x": 65, "y": 75},
            {"role": "RB", "x": 85, "y": 72},
            {"role": "CM", "x": 30, "y": 52},
            {"role": "CM", "x": 50, "y": 48},
            {"role": "CM", "x": 70, "y": 52},
            {"role": "LW", "x": 18, "y": 25},
            {"role": "ST", "x": 50, "y": 18},
            {"role": "RW", "x": 82, "y": 25},
        ]
    },
    "4-2-3-1": {
        "positions": [
            {"role": "GK", "x": 50, "y": 92},
            {"role": "LB", "x": 15, "y": 72},
            {"role": "CB", "x": 35, "y": 75},
            {"role": "CB", "x": 65, "y": 75},
            {"role": "RB", "x": 85, "y": 72},
            {"role": "CDM", "x": 35, "y": 55},
            {"role": "CDM", "x": 65, "y": 55},
            {"role": "CAM", "x": 50, "y": 38},
            {"role": "LW", "x": 20, "y": 30},
            {"role": "ST", "x": 50, "y": 15},
            {"role": "RW", "x": 80, "y": 30},
        ]
    },
    "4-4-2": {
        "positions": [
            {"role": "GK", "x": 50, "y": 92},
            {"role": "LB", "x": 15, "y": 72},
            {"role": "CB", "x": 35, "y": 75},
            {"role": "CB", "x": 65, "y": 75},
            {"role": "RB", "x": 85, "y": 72},
            {"role": "LM", "x": 15, "y": 50},
            {"role": "CM", "x": 38, "y": 50},
            {"role": "CM", "x": 62, "y": 50},
            {"role": "RM", "x": 85, "y": 50},
            {"role": "ST", "x": 38, "y": 20},
            {"role": "ST", "x": 62, "y": 20},
        ]
    }
}

print(f"✅ {len(FORMATIONS)} formation templates defined")

✅ 3 formation templates defined


In [14]:
# Quiz generation functions
def generate_quiz(players, team_name, country_code, match_info, formation, player_order):
    """
    Generate a Starting11 quiz JSON.
    
    Args:
        players: List of player dicts from scraper (first 11 = starting lineup)
        team_name: e.g., "Argentina"
        country_code: e.g., "ARG"
        match_info: dict with tournament, date, opponent, etc.
        formation: e.g., "4-3-3"
        player_order: List of {jersey, role} dicts mapping players to positions
    """
    quiz = {
        "answer": {
            "country": team_name,
            "country_code": country_code
        },
        "match": match_info,
        "formation": formation,
        "players": []
    }
    
    # Create jersey -> position mapping
    positions = FORMATIONS[formation]["positions"]
    jersey_to_idx = {p["jersey"]: i for i, p in enumerate(player_order)}
    
    for p in players[:11]:
        player_data = {
            "jersey": p["jersey"],
            "name": p["name"],
            "player_id": p["player_id"],
            "club": {
                "name": p["club_name"],
                "club_id": p["club_id"],
                "badge_url": f"https://tmssl.akamaized.net/images/wappen/head/{p['club_id']}.png"
            }
        }
        
        # Add position if we have mapping
        if p["jersey"] in jersey_to_idx:
            idx = jersey_to_idx[p["jersey"]]
            pos = positions[idx]
            player_data["position"] = {
                "role": pos["role"],
                "x": pos["x"],
                "y": pos["y"]
            }
        
        quiz["players"].append(player_data)
    
    return quiz

print("✅ Quiz generator defined")

✅ Quiz generator defined


In [15]:
# HTML visualization generator
def generate_pitch_html(quiz, output_path):
    """Generate an HTML visualization of the quiz."""
    html = '''<!DOCTYPE html>
<html>
<head>
    <title>Starting11 - ''' + quiz['match']['tournament'] + '''</title>
    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; }
        body { font-family: -apple-system, BlinkMacSystemFont, sans-serif; background: #1a1a2e; color: white; }
        .container { max-width: 900px; margin: 0 auto; padding: 20px; }
        h1 { text-align: center; margin-bottom: 10px; }
        .subtitle { text-align: center; color: #888; margin-bottom: 30px; }
        .pitch {
            position: relative; width: 100%; aspect-ratio: 1.4;
            background: linear-gradient(to bottom, #2d5a27 0%, #3d7a37 50%, #2d5a27 100%);
            border-radius: 8px; border: 3px solid rgba(255,255,255,0.8); overflow: hidden;
        }
        .pitch::before {
            content: ''; position: absolute; top: 50%; left: 50%;
            transform: translate(-50%, -50%); width: 100px; height: 100px;
            border: 2px solid rgba(255,255,255,0.4); border-radius: 50%;
        }
        .pitch::after {
            content: ''; position: absolute; top: 50%; left: 0; right: 0;
            height: 2px; background: rgba(255,255,255,0.4);
        }
        .penalty-area { position: absolute; left: 50%; transform: translateX(-50%); width: 44%; height: 16%; border: 2px solid rgba(255,255,255,0.4); }
        .penalty-top { top: 0; border-top: none; }
        .penalty-bottom { bottom: 0; border-bottom: none; }
        .player {
            position: absolute; display: flex; flex-direction: column; align-items: center;
            transform: translate(-50%, -50%); transition: transform 0.2s;
        }
        .player:hover { transform: translate(-50%, -50%) scale(1.1); z-index: 10; }
        .badge { width: 48px; height: 48px; background: white; border-radius: 8px; padding: 3px; box-shadow: 0 2px 8px rgba(0,0,0,0.4); }
        .badge img { width: 100%; height: 100%; object-fit: contain; }
        .jersey { margin-top: 4px; background: rgba(0,0,0,0.8); padding: 2px 8px; border-radius: 10px; font-size: 11px; font-weight: bold; }
        .answer-box { margin-top: 30px; text-align: center; }
        .answer-box input { padding: 12px 20px; font-size: 18px; border: none; border-radius: 8px; width: 300px; }
        .answer-box button { padding: 12px 30px; font-size: 18px; background: #4CAF50; color: white; border: none; border-radius: 8px; cursor: pointer; margin-left: 10px; }
        .formation-label { position: absolute; bottom: 10px; right: 10px; background: rgba(0,0,0,0.6); padding: 4px 10px; border-radius: 4px; font-size: 12px; }
    </style>
</head>
<body>
    <div class="container">
        <h1>⚽ Starting 11</h1>
        <p class="subtitle">''' + quiz['match']['tournament'] + ' ' + quiz['match']['stage'] + ''' • Guess the National Team</p>
        <div class="pitch">
            <div class="penalty-area penalty-top"></div>
            <div class="penalty-area penalty-bottom"></div>
'''
    
    for p in quiz['players']:
        pos = p.get('position', {})
        x, y = pos.get('x', 50), pos.get('y', 50)
        badge_url = p['club']['badge_url']
        html += f'''            <div class="player" style="left: {x}%; top: {y}%;">
                <div class="badge"><img src="{badge_url}" alt="{p['club']['name']}"></div>
                <span class="jersey">#{p['jersey']}</span>
            </div>
'''
    
    answer = quiz['answer']['country']
    html += f'''            <div class="formation-label">{quiz.get('formation', '')}</div>
        </div>
        <div class="answer-box">
            <input type="text" placeholder="Which national team is this?" id="guess">
            <button onclick="checkAnswer()">Submit</button>
        </div>
    </div>
    <script>
        function checkAnswer() {{
            const guess = document.getElementById('guess').value.toLowerCase();
            if (guess.includes('{answer.lower()[:6]}')) {{
                alert('🎉 Correct! This is {answer}\\'s starting 11!');
            }} else {{
                alert('❌ Try again!');
            }}
        }}
        document.getElementById('guess').addEventListener('keypress', e => {{ if (e.key === 'Enter') checkAnswer(); }});
    </script>
</body>
</html>'''
    
    with open(output_path, 'w') as f:
        f.write(html)
    print(f"✅ Saved: {output_path}")

print("✅ HTML generator defined")

✅ HTML generator defined


---
## Scrape 2022 World Cup Final

In [16]:
# Scrape the 2022 World Cup Final
match_url = "https://www.transfermarkt.com/argentina_france/aufstellung/spielbericht/3975879"
print("Scraping 2022 World Cup Final lineup...\n")

players = scrape_match_lineup(match_url)

if players:
    print(f"\n✅ Found {len(players)} players!\n")
    print("--- Argentina Starting 11 ---")
    for p in players[:11]:
        print(f"  #{p['jersey']:2} {p['name']:<25} - {p['club_name']}")
    print("\n--- France Starting 11 ---")
    for p in players[11:22]:
        print(f"  #{p['jersey']:2} {p['name']:<25} - {p['club_name']}")

Scraping 2022 World Cup Final lineup...

  Waiting 3s before request...
  Fetching: https://www.transfermarkt.com/argentina_france/aufstellung/spielbericht/3975879

✅ Found 50 players!

--- Argentina Starting 11 ---
  #23 Emiliano Martínez         - Aston Villa
  #13 Cristian Romero           - Tottenham Hotspur
  #19 Nicolás Otamendi          - SL Benfica
  # 3 Nicolás Tagliafico        - Olympique Lyon
  #26 Nahuel Molina             - Atlético de Madrid
  #24 Enzo Fernández            - SL Benfica
  # 7 Rodrigo De Paul           - Atlético de Madrid
  #20 Alexis Mac Allister       - Brighton & Hove Albion
  #11 Ángel Di María            - Juventus FC
  #10 Lionel Messi              - Paris Saint-Germain
  # 9 Julián Alvarez            - Manchester City

--- France Starting 11 ---
  # 1 Hugo Lloris               - Tottenham Hotspur
  # 4 Raphaël Varane            - Manchester United
  #18 Dayot Upamecano           - Bayern Munich
  #22 Theo Hernández            - AC Milan
  # 5 Jules

In [17]:
# Define player positions for the 2022 World Cup Final
# (Based on Transfermarkt's formation visualization)

ARGENTINA_2022_FINAL = {
    "formation": "4-3-3",
    "player_order": [
        {"jersey": 23, "role": "GK"},      # Martínez
        {"jersey": 3, "role": "LB"},       # Tagliafico
        {"jersey": 19, "role": "CB"},      # Otamendi
        {"jersey": 13, "role": "CB"},      # Romero
        {"jersey": 26, "role": "RB"},      # Molina
        {"jersey": 24, "role": "CM"},      # Enzo Fernández
        {"jersey": 7, "role": "CM"},       # De Paul
        {"jersey": 20, "role": "CM"},      # Mac Allister
        {"jersey": 11, "role": "LW"},      # Di María
        {"jersey": 9, "role": "ST"},       # Álvarez
        {"jersey": 10, "role": "RW"},      # Messi
    ]
}

FRANCE_2022_FINAL = {
    "formation": "4-2-3-1",
    "player_order": [
        {"jersey": 1, "role": "GK"},       # Lloris
        {"jersey": 22, "role": "LB"},      # Theo Hernández
        {"jersey": 18, "role": "CB"},      # Upamecano
        {"jersey": 4, "role": "CB"},       # Varane
        {"jersey": 5, "role": "RB"},       # Koundé
        {"jersey": 8, "role": "CDM"},      # Tchouaméni
        {"jersey": 14, "role": "CDM"},     # Rabiot
        {"jersey": 7, "role": "CAM"},      # Griezmann
        {"jersey": 10, "role": "LW"},      # Mbappé
        {"jersey": 9, "role": "ST"},       # Giroud
        {"jersey": 11, "role": "RW"},      # Dembélé
    ]
}

print("✅ Position mappings defined")

✅ Position mappings defined


In [18]:
# Generate quizzes
argentina_quiz = generate_quiz(
    players[:11],
    team_name="Argentina",
    country_code="ARG",
    match_info={
        "tournament": "World Cup 2022",
        "stage": "Final",
        "date": "2022-12-18",
        "opponent": "France",
        "result": "3-3 (4-2 pens)",
        "venue": "Lusail Stadium"
    },
    formation="4-3-3",
    player_order=ARGENTINA_2022_FINAL["player_order"]
)

france_quiz = generate_quiz(
    players[11:22],
    team_name="France",
    country_code="FRA",
    match_info={
        "tournament": "World Cup 2022",
        "stage": "Final",
        "date": "2022-12-18",
        "opponent": "Argentina",
        "result": "3-3 (2-4 pens)",
        "venue": "Lusail Stadium"
    },
    formation="4-2-3-1",
    player_order=FRANCE_2022_FINAL["player_order"]
)

print("Argentina Quiz:")
for p in argentina_quiz['players']:
    pos = p.get('position', {})
    print(f"  #{p['jersey']:2} {p['name']:<22} {pos.get('role', '?'):>4} @ ({pos.get('x', '?'):>2}, {pos.get('y', '?'):>2})")

Argentina Quiz:
  #23 Emiliano Martínez        GK @ (50, 92)
  #13 Cristian Romero          CB @ (65, 75)
  #19 Nicolás Otamendi         CB @ (35, 75)
  # 3 Nicolás Tagliafico       LB @ (15, 72)
  #26 Nahuel Molina            RB @ (85, 72)
  #24 Enzo Fernández           CM @ (30, 52)
  # 7 Rodrigo De Paul          CM @ (50, 48)
  #20 Alexis Mac Allister      CM @ (70, 52)
  #11 Ángel Di María           LW @ (18, 25)
  #10 Lionel Messi             RW @ (82, 25)
  # 9 Julián Alvarez           ST @ (50, 18)


In [19]:
# Save quizzes to JSON
os.makedirs('../quizzes/starting11/wc2022', exist_ok=True)

with open('../quizzes/starting11/wc2022/final_argentina.json', 'w') as f:
    json.dump(argentina_quiz, f, indent=2)
print("✅ Saved: quizzes/starting11/wc2022/final_argentina.json")

with open('../quizzes/starting11/wc2022/final_france.json', 'w') as f:
    json.dump(france_quiz, f, indent=2)
print("✅ Saved: quizzes/starting11/wc2022/final_france.json")

✅ Saved: quizzes/starting11/wc2022/final_argentina.json
✅ Saved: quizzes/starting11/wc2022/final_france.json


In [20]:
# Investigate: Can we scrape positions from the match INDEX page?
# The index page has a visual formation diagram

index_url = "https://www.transfermarkt.com/argentina_france/index/spielbericht/3975879"
soup = fetch_page(index_url, delay=True)

if soup:
    # Save for inspection
    with open('debug_index_page.html', 'w', encoding='utf-8') as f:
        f.write(soup.prettify())
    
    # Look for the formation diagram elements
    print("--- Looking for formation/lineup elements ---")
    
    # Check for elements with positioning styles
    positioned = soup.find_all(style=re.compile(r'(top|left):\s*\d+'))
    print(f"Elements with position styles: {len(positioned)}")
    
    # Look for lineup container
    lineup_containers = soup.find_all(['div', 'ul'], class_=re.compile(r'aufstellung|lineup|field', re.I))
    print(f"Lineup containers: {len(lineup_containers)}")
    
    # Look for player items in the formation
    # They often have jersey numbers and position classes
    formation_items = soup.find_all(['li', 'div'], class_=re.compile(r'^[A-Z]{2,3}$'))  # Like "GK", "CB", "LW"
    print(f"Position-class items: {len(formation_items)}")
    
    # Check for data attributes
    data_position = soup.find_all(attrs={'data-position': True})
    print(f"Elements with data-position: {len(data_position)}")
    
    # Look for the specific formation text
    formation_text = soup.find(string=re.compile(r'Starting Line-up:\s*\d+-\d+'))
    if formation_text:
        print(f"\nFormation found: {formation_text.strip()}")

  Waiting 3s before request...
  Fetching: https://www.transfermarkt.com/argentina_france/index/spielbericht/3975879
--- Looking for formation/lineup elements ---
Elements with position styles: 74
Lineup containers: 11
Position-class items: 0
Elements with data-position: 0

Formation found: Starting Line-up: 4-3-3 Attacking


In [21]:
# Deep dive: Find the actual lineup visualization structure
if soup:
    # The formation visual usually has player jersey numbers positioned on a pitch
    # Look for elements containing just jersey numbers (1-26)
    
    print("--- Searching for lineup wrapper ---")
    
    # Try finding by common wrapper classes
    wrappers = soup.find_all('div', class_=re.compile(r'(large-6|columns|lineup|formation)', re.I))
    print(f"Potential wrappers: {len(wrappers)}")
    
    # Look for the lineup sections (home/away)
    sections = soup.find_all('div', class_='large-6')
    print(f"large-6 sections (likely home/away): {len(sections)}")
    
    for i, section in enumerate(sections[:4]):
        # Check if this section contains lineup info
        has_lineup = section.find(string=re.compile(r'Starting Line-up'))
        if has_lineup:
            print(f"\n=== Section {i} has lineup ===")
            
            # Find all the player entries
            # Look for links to player profiles within this section
            player_entries = section.find_all('a', href=re.compile(r'/profil/spieler/'))
            print(f"Player links: {len(player_entries)}")
            
            # Check for jersey number elements
            jersey_spans = section.find_all(['span', 'div'], string=re.compile(r'^\\d{1,2}$'))
            print(f"Jersey number elements: {len(jersey_spans)}")
            
            # Look at the structure
            formation_div = section.find('div', class_=re.compile(r'aufstellung'))
            if formation_div:
                print(f"Found aufstellung div with classes: {formation_div.get('class')}")
                # Get first few children
                children = formation_div.find_all(recursive=False)[:5]
                for c in children:
                    print(f"  Child: <{c.name}> class={c.get('class')}")

--- Searching for lineup wrapper ---
Potential wrappers: 43
large-6 sections (likely home/away): 2

=== Section 0 has lineup ===
Player links: 26
Jersey number elements: 0
Found aufstellung div with classes: ['unterueberschrift', 'aufstellung-unterueberschrift-mannschaft']
  Child: <div> class=None
  Child: <div> class=None

=== Section 1 has lineup ===
Player links: 24
Jersey number elements: 0
Found aufstellung div with classes: ['unterueberschrift', 'aufstellung-unterueberschrift-mannschaft', 'aufstellung-bordertop-small']
  Child: <div> class=None
  Child: <div> class=None


In [23]:
# Analyze the positioned elements - these should contain player positions
if soup:
    print("--- Analyzing positioned elements ---\\n")
    
    positioned = soup.find_all(style=re.compile(r'(top|left):\s*\d+'))
    
    # Filter for elements that might be player markers
    player_positions = []
    
    for elem in positioned:
        style = elem.get('style', '')
        
        # Extract top and left values
        top_match = re.search(r'top:\s*(\d+)', style)
        left_match = re.search(r'left:\s*(\d+)', style)
        
        if top_match and left_match:
            top = int(top_match.group(1))
            left = int(left_match.group(1))
            
            # Get any text content (might be jersey number)
            text = elem.get_text(strip=True)
            
            # Check for player link inside
            player_link = elem.find('a', href=re.compile(r'/profil/spieler/'))
            player_name = None
            player_id = None
            
            if player_link:
                href = player_link.get('href', '')
                match = re.search(r'/([^/]+)/profil/spieler/(\\d+)', href)
                if match:
                    player_name = match.group(1).replace('-', ' ').title()
                    player_id = match.group(2)
            
            # Also check parent for player info
            if not player_link:
                parent = elem.find_parent(style=re.compile(r'(top|left)'))
                if parent:
                    player_link = parent.find('a', href=re.compile(r'/profil/spieler/'))
            
            if player_id or text.isdigit():
                player_positions.append({
                    'top': top,
                    'left': left,
                    'text': text[:20] if text else '',
                    'player_id': player_id,
                    'player_name': player_name,
                    'tag': elem.name,
                    'class': elem.get('class')
                })
    
    print(f"Found {len(player_positions)} positioned elements with player info\\n")
    
    # Sort by top (y position) to see the formation
    player_positions.sort(key=lambda x: (x['top'], x['left']))
    
    for p in player_positions[:20]:
        print(f"  top:{p['top']:3} left:{p['left']:3} | {p['text']:<10} | {p['player_name'] or 'no name'}")

--- Analyzing positioned elements ---\n
Found 0 positioned elements with player info\n


In [24]:
# Alternative: Look at lineup containers more closely
if soup:
    print("--- Examining lineup containers ---\\n")
    
    lineup_containers = soup.find_all('div', class_=re.compile(r'aufstellung'))
    
    for i, container in enumerate(lineup_containers[:5]):
        classes = container.get('class', [])
        print(f"Container {i}: {classes}")
        
        # Look for nested divs with positioning
        nested = container.find_all('div', style=re.compile(r'top.*left|left.*top'))
        if nested:
            print(f"  -> Has {len(nested)} positioned children")
            for n in nested[:3]:
                style = n.get('style', '')[:50]
                text = n.get_text(strip=True)[:30]
                print(f"     style: {style}... text: {text}")
        print()
    
    # Also try finding the pitch/field container
    print("--- Looking for pitch/field visual ---")
    pitch = soup.find('div', class_=re.compile(r'pitch|field|spielfeld', re.I))
    if pitch:
        print(f"Found pitch: {pitch.get('class')}")
    else:
        # Look for the actual lineup graphic
        lineup_graphic = soup.find('div', class_='aufstellung-grafik')
        if lineup_graphic:
            print(f"Found lineup graphic!")
            positioned_in_graphic = lineup_graphic.find_all(style=re.compile(r'top.*left'))
            print(f"Positioned elements in graphic: {len(positioned_in_graphic)}")

--- Examining lineup containers ---\n
Container 0: ['large-6', 'columns', 'aufstellung-box']
  -> Has 19 positioned children
     style: position: absolute; top: 0; bottom: 0; left: 0; ri... text: 23Martínez19Otamendi13Romero3T
     style: top: 80%; left: 40%;... text: 23Martínez
     style: top: 63%; left: 28%;... text: 19Otamendi

Container 1: ['unterueberschrift', 'aufstellung-unterueberschrift-mannschaft']

Container 2: ['large-7', 'aufstellung-vereinsseite', 'columns', 'small-12', 'unterueberschrift', 'formation-subtitle']

Container 3: ['large-5', 'aufstellung-vereinsseite', 'columns', 'small-12', 'hide-for-small', 'unterueberschrift']

Container 4: ['large-7', 'columns', 'small-12', 'aufstellung-vereinsseite']
  -> Has 19 positioned children
     style: position: absolute; top: 0; bottom: 0; left: 0; ri... text: 23Martínez19Otamendi13Romero3T
     style: top: 80%; left: 40%;... text: 23Martínez
     style: top: 63%; left: 28%;... text: 19Otamendi

--- Looking for pitch/field vis

In [25]:
# Extract player positions from the aufstellung-box containers
def extract_lineup_positions(soup):
    """
    Extract player positions from Transfermarkt match index page.
    Returns list of dicts with jersey, name, top%, left% for each team.
    """
    teams = []
    
    # Find the aufstellung-box containers (one per team)
    lineup_boxes = soup.find_all('div', class_='aufstellung-box')
    print(f"Found {len(lineup_boxes)} lineup boxes")
    
    for box_idx, box in enumerate(lineup_boxes):
        team_players = []
        
        # Find positioned elements with percentage-based positions
        positioned = box.find_all('div', style=re.compile(r'top:\s*\d+%.*left:\s*\d+%'))
        
        for elem in positioned:
            style = elem.get('style', '')
            
            # Extract percentages
            top_match = re.search(r'top:\s*(\d+)%', style)
            left_match = re.search(r'left:\s*(\d+)%', style)
            
            if top_match and left_match:
                top_pct = int(top_match.group(1))
                left_pct = int(left_match.group(1))
                
                # Get text content (jersey + name)
                text = elem.get_text(strip=True)
                
                # Parse jersey number (first digits)
                jersey_match = re.match(r'(\d+)', text)
                if jersey_match:
                    jersey = int(jersey_match.group(1))
                    name = text[len(jersey_match.group(1)):].strip()
                    
                    team_players.append({
                        'jersey': jersey,
                        'name': name,
                        'top_pct': top_pct,
                        'left_pct': left_pct
                    })
        
        if team_players:
            # Sort by position (GK first = highest top%)
            team_players.sort(key=lambda x: (-x['top_pct'], x['left_pct']))
            teams.append(team_players)
    
    return teams

# Test it
if soup:
    teams = extract_lineup_positions(soup)
    
    for i, team in enumerate(teams):
        print(f"\\n=== Team {i+1} ({len(team)} players) ===")
        for p in team:
            print(f"  #{p['jersey']:2} {p['name']:<20} @ top:{p['top_pct']:2}% left:{p['left_pct']:2}%")

Found 1 lineup boxes
\n=== Team 1 (9 players) ===
  #23 Martínez             @ top:80% left:40%
  #19 Otamendi             @ top:63% left:28%
  #26 Molina               @ top:61% left:73%
  #24 Fernández            @ top:39% left:40%
  #20 Mac Allister         @ top:28% left:27%
  # 7 De Paul              @ top:28% left:53%
  #11 Di María             @ top:10% left:15%
  #10 Messi                @ top:10% left:65%
  # 9 Alvarez              @ top: 3% left:40%


In [26]:
# Debug: Find the missing players
if soup:
    box = soup.find('div', class_='aufstellung-box')
    
    # Get ALL positioned divs, regardless of style format
    all_positioned = box.find_all('div', style=re.compile(r'%'))
    print(f"Total elements with % in style: {len(all_positioned)}\\n")
    
    # Check each one
    for elem in all_positioned:
        style = elem.get('style', '')
        text = elem.get_text(strip=True)
        
        # Only show ones that look like player entries (start with digits)
        if text and re.match(r'^\d+[A-Z]', text):
            print(f"Style: {style[:60]}...")
            print(f"Text: {text[:30]}")
            
            # Check what patterns exist
            has_top = 'top:' in style or 'top :' in style
            has_left = 'left:' in style or 'left :' in style
            print(f"Has top: {has_top}, Has left: {has_left}\\n")

Total elements with % in style: 24\n
Style: top: 80%; left: 40%;...
Text: 23Martínez
Has top: True, Has left: True\n
Style: top: 63%; left: 28%;...
Text: 19Otamendi
Has top: True, Has left: True\n
Style: top: 63%; left: 52.5%;...
Text: 13Romero
Has top: True, Has left: True\n
Style: top: 61%; left: 7.5%;...
Text: 3Tagliafico
Has top: True, Has left: True\n
Style: top: 61%; left: 73%;...
Text: 26Molina
Has top: True, Has left: True\n
Style: top: 39%; left: 40%;...
Text: 24Fernández
Has top: True, Has left: True\n
Style: top: 28%; left: 53%;...
Text: 7De Paul
Has top: True, Has left: True\n
Style: top: 28%; left: 27%;...
Text: 20Mac Allister
Has top: True, Has left: True\n
Style: top: 10%; left: 15%;...
Text: 11Di María
Has top: True, Has left: True\n
Style: top: 10%; left: 65%;...
Text: 10Messi
Has top: True, Has left: True\n
Style: top: 3%; left: 40%;...
Text: 9Alvarez
Has top: True, Has left: True\n


In [27]:
# Improved extraction - more flexible regex
def extract_lineup_positions_v2(soup):
    """Extract player positions with more flexible pattern matching."""
    teams = []
    
    lineup_boxes = soup.find_all('div', class_='aufstellung-box')
    print(f"Found {len(lineup_boxes)} lineup boxes")
    
    for box in lineup_boxes:
        team_players = []
        
        # Get all divs with style containing percentages
        all_divs = box.find_all('div', style=True)
        
        for elem in all_divs:
            style = elem.get('style', '')
            text = elem.get_text(strip=True)
            
            # Skip if doesn't look like a player entry
            if not text or not re.match(r'^\d+[A-Za-z]', text):
                continue
            
            # More flexible extraction - handle various formats
            top_match = re.search(r'top\s*:\s*(\d+)%', style)
            left_match = re.search(r'left\s*:\s*(\d+)%', style)
            
            if top_match and left_match:
                top_pct = int(top_match.group(1))
                left_pct = int(left_match.group(1))
                
                # Parse jersey number
                jersey_match = re.match(r'(\d+)', text)
                if jersey_match:
                    jersey = int(jersey_match.group(1))
                    name = text[len(jersey_match.group(1)):].strip()
                    
                    # Avoid duplicates
                    if not any(p['jersey'] == jersey for p in team_players):
                        team_players.append({
                            'jersey': jersey,
                            'name': name,
                            'top_pct': top_pct,
                            'left_pct': left_pct
                        })
        
        if team_players:
            team_players.sort(key=lambda x: (-x['top_pct'], x['left_pct']))
            teams.append(team_players)
    
    return teams

# Test v2
if soup:
    teams = extract_lineup_positions_v2(soup)
    
    for i, team in enumerate(teams):
        print(f"\\n=== Team {i+1} ({len(team)} players) ===")
        for p in team:
            print(f"  #{p['jersey']:2} {p['name']:<20} @ top:{p['top_pct']:2}% left:{p['left_pct']:2}%")

Found 1 lineup boxes
\n=== Team 1 (9 players) ===
  #23 Martínez             @ top:80% left:40%
  #19 Otamendi             @ top:63% left:28%
  #26 Molina               @ top:61% left:73%
  #24 Fernández            @ top:39% left:40%
  #20 Mac Allister         @ top:28% left:27%
  # 7 De Paul              @ top:28% left:53%
  #11 Di María             @ top:10% left:15%
  #10 Messi                @ top:10% left:65%
  # 9 Alvarez              @ top: 3% left:40%


In [28]:
# FIXED: Handle decimal percentages and find both teams
def extract_lineup_positions_v3(soup):
    """Extract player positions - handles decimal percentages."""
    teams = []
    
    # Try multiple container classes
    lineup_boxes = soup.find_all('div', class_='aufstellung-box')
    if len(lineup_boxes) < 2:
        # Also try the large-6 columns
        lineup_boxes = soup.find_all('div', class_=re.compile(r'large-6.*columns|columns.*large-6'))
    
    print(f"Found {len(lineup_boxes)} potential lineup containers")
    
    for box in lineup_boxes:
        team_players = []
        
        all_divs = box.find_all('div', style=True)
        
        for elem in all_divs:
            style = elem.get('style', '')
            text = elem.get_text(strip=True)
            
            if not text or not re.match(r'^\d+[A-Za-z]', text):
                continue
            
            # FIXED: Handle decimals like 52.5%
            top_match = re.search(r'top\s*:\s*([\d.]+)%', style)
            left_match = re.search(r'left\s*:\s*([\d.]+)%', style)
            
            if top_match and left_match:
                top_pct = float(top_match.group(1))
                left_pct = float(left_match.group(1))
                
                jersey_match = re.match(r'(\d+)', text)
                if jersey_match:
                    jersey = int(jersey_match.group(1))
                    name = text[len(jersey_match.group(1)):].strip()
                    
                    if not any(p['jersey'] == jersey for p in team_players):
                        team_players.append({
                            'jersey': jersey,
                            'name': name,
                            'top_pct': top_pct,
                            'left_pct': left_pct
                        })
        
        # Only add if we have 11 players (starting lineup)
        if len(team_players) == 11:
            team_players.sort(key=lambda x: (-x['top_pct'], x['left_pct']))
            teams.append(team_players)
    
    return teams

# Test v3
if soup:
    teams = extract_lineup_positions_v3(soup)
    
    for i, team in enumerate(teams):
        print(f"\\n=== Team {i+1} ({len(team)} players) ===")
        for p in team:
            print(f"  #{p['jersey']:2} {p['name']:<20} @ top:{p['top_pct']:5.1f}% left:{p['left_pct']:5.1f}%")

Found 2 potential lineup containers
\n=== Team 1 (11 players) ===
  #23 Martínez             @ top: 80.0% left: 40.0%
  #19 Otamendi             @ top: 63.0% left: 28.0%
  #13 Romero               @ top: 63.0% left: 52.5%
  # 3 Tagliafico           @ top: 61.0% left:  7.5%
  #26 Molina               @ top: 61.0% left: 73.0%
  #24 Fernández            @ top: 39.0% left: 40.0%
  #20 Mac Allister         @ top: 28.0% left: 27.0%
  # 7 De Paul              @ top: 28.0% left: 53.0%
  #11 Di María             @ top: 10.0% left: 15.0%
  #10 Messi                @ top: 10.0% left: 65.0%
  # 9 Alvarez              @ top:  3.0% left: 40.0%
\n=== Team 2 (11 players) ===
  # 1 Lloris               @ top: 80.0% left: 40.0%
  #18 Upamecano            @ top: 63.0% left: 28.0%
  # 4 Varane               @ top: 63.0% left: 52.5%
  #22 Hernández            @ top: 61.0% left:  7.5%
  # 5 Koundé               @ top: 61.0% left: 73.0%
  # 8 Tchouaméni           @ top: 43.0% left: 28.0%
  #14 Rabiot        

In [29]:
# Find France's lineup - look at all large-6 containers
if soup:
    print("--- Looking for both team lineups ---\\n")
    
    # The page has two large-6 columns side by side
    sections = soup.find_all('div', class_='large-6')
    print(f"Found {len(sections)} large-6 sections")
    
    for i, section in enumerate(sections):
        # Check if this section has positioned player divs
        positioned = section.find_all('div', style=re.compile(r'top.*%.*left|left.*%.*top'))
        player_texts = []
        for p in positioned:
            text = p.get_text(strip=True)
            if text and re.match(r'^\d+[A-Za-z]', text):
                player_texts.append(text[:15])
        
        if len(player_texts) >= 11:
            print(f"\\nSection {i}: Has {len(player_texts)} player entries")
            print(f"  First 3: {player_texts[:3]}")
            print(f"  Classes: {section.get('class')}")

--- Looking for both team lineups ---\n
Found 2 large-6 sections
\nSection 0: Has 11 player entries
  First 3: ['23Martínez', '19Otamendi', '13Romero']
  Classes: ['large-6', 'columns', 'aufstellung-box']
\nSection 1: Has 11 player entries
  First 3: ['1Lloris', '18Upamecano', '4Varane']
  Classes: ['large-6', 'columns']


In [ ]:
# FINAL: Complete automated scraper combining lineup + positions
def scrape_match_complete(match_id):
    """
    Scrape a complete match with players, clubs, and positions.
    
    Combines:
    - /aufstellung/ page: Player IDs, clubs, badge URLs
    - /index/ page: Pitch positions (x, y percentages)
    
    Returns dict with both teams' data, ready for quiz generation.
    """
    base = "https://www.transfermarkt.com"
    
    # We need the team slug from the URL - for now use a placeholder
    # In practice, you'd get this from the match listing
    lineup_url = f"{base}/spielbericht/aufstellung/spielbericht/{match_id}"
    index_url = f"{base}/spielbericht/index/spielbericht/{match_id}"
    
    result = {'match_id': match_id, 'teams': []}
    
    # Step 1: Get positions from index page
    print("Step 1: Fetching positions from index page...")
    index_soup = fetch_page(index_url, delay=True)
    
    positions_by_team = []
    if index_soup:
        positions_by_team = extract_lineup_positions_v3(index_soup)
        print(f"  Got positions for {len(positions_by_team)} teams")
    
    # Step 2: Get player details from lineup page  
    print("\\nStep 2: Fetching player details from lineup page...")
    lineup_soup = fetch_page(lineup_url, delay=True)
    
    players_list = []
    if lineup_soup:
        players_list = scrape_match_lineup(lineup_url)  # Uses existing function
        print(f"  Got {len(players_list)} total players")
    
    # Step 3: Merge the data
    print("\\nStep 3: Merging positions with player details...")
    
    # Split players into teams (first 11 = team 1, etc.)
    # The lineup page returns: team1 starters, team2 starters, team1 subs, team2 subs
    team1_players = players_list[:11] if len(players_list) >= 11 else []
    team2_players = players_list[11:22] if len(players_list) >= 22 else []
    
    for team_idx, (players, positions) in enumerate(zip([team1_players, team2_players], positions_by_team)):
        team_data = {'players': []}
        
        # Create jersey -> position mapping
        pos_by_jersey = {p['jersey']: p for p in positions}
        
        for player in players:
            jersey = player['jersey']
            pos = pos_by_jersey.get(jersey, {})
            
            team_data['players'].append({
                'jersey': jersey,
                'name': player['name'],
                'player_id': player['player_id'],
                'club': {
                    'name': player['club_name'],
                    'club_id': player['club_id'],
                    'badge_url': f"https://tmssl.akamaized.net/images/wappen/head/{player['club_id']}.png"
                },
                'position': {
                    'x': pos.get('left_pct', 50),
                    'y': pos.get('top_pct', 50)
                }
            })
        
        result['teams'].append(team_data)
    
    return result

print("✅ Complete scraper defined!")

In [ ]:
# Test the complete scraper with the World Cup Final
# Note: We already have the soup objects from earlier, let's use them

print("=== Testing Complete Pipeline ===\\n")

# We have: lineup page players + index page positions
# Let's merge them properly

team1_players = players[:11]  # Argentina from lineup page
team2_players = players[11:22]  # France from lineup page

team1_positions = teams[0]  # Argentina positions from index page
team2_positions = teams[1]  # France positions from index page

def merge_player_data(players_with_clubs, positions):
    """Merge player club data with position data."""
    pos_by_jersey = {p['jersey']: p for p in positions}
    merged = []
    
    for player in players_with_clubs:
        jersey = player['jersey']
        pos = pos_by_jersey.get(jersey, {})
        
        merged.append({
            'jersey': jersey,
            'name': player['name'],
            'player_id': player['player_id'],
            'club': {
                'name': player['club_name'],
                'club_id': player['club_id'],
                'badge_url': f"https://tmssl.akamaized.net/images/wappen/head/{player['club_id']}.png"
            },
            'position': {
                'x': pos.get('left_pct', 50),
                'y': pos.get('top_pct', 50)
            }
        })
    
    return merged

argentina_merged = merge_player_data(team1_players, team1_positions)
france_merged = merge_player_data(team2_players, team2_positions)

print("Argentina (merged):")
for p in argentina_merged:
    print(f"  #{p['jersey']:2} {p['name']:<22} {p['club']['name']:<25} @ ({p['position']['x']:5.1f}, {p['position']['y']:5.1f})")

print("\\nFrance (merged):")
for p in france_merged:
    print(f"  #{p['jersey']:2} {p['name']:<22} {p['club']['name']:<25} @ ({p['position']['x']:5.1f}, {p['position']['y']:5.1f})")

In [22]:
# Generate HTML visualizations
generate_pitch_html(argentina_quiz, 'starting11_argentina.html')
generate_pitch_html(france_quiz, 'starting11_france.html')

print("\n🎉 Done! Open the HTML files in a browser to preview.")

✅ Saved: starting11_argentina.html
✅ Saved: starting11_france.html

🎉 Done! Open the HTML files in a browser to preview.
